CELL 1 — Install dependencies + Mount Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, os

print("📦 Installing nnU-Net...")
subprocess.run(['pip', 'install', 'nnunet', '-q'], check=True)

print("📦 Installing API server dependencies...")
subprocess.run(['pip', 'install', 'fastapi', 'uvicorn', 'python-multipart', 'pyngrok', '-q'], check=True)

print("📦 Installing ML dependencies...")
subprocess.run(['pip', 'install', '-q', '-U', 'transformers', 'accelerate', 'bitsandbytes'], check=True)
subprocess.run(['pip', 'install', '-q', 'pillow'], check=True)

os.environ['nnUNet_raw_data_base'] = '/content/nnunet_raw'
os.environ['nnUNet_preprocessed']  = '/content/nnunet_preprocessed'
os.environ['RESULTS_FOLDER']       = '/content/nnunet_results'

for path in ['/content/nnunet_raw', '/content/nnunet_preprocessed',
             '/content/nnunet_results', '/content/api_uploads']:
    os.makedirs(path, exist_ok=True)

import torch
print(f"\n✅ Setup complete")
print(f"   GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mounted at /content/drive
📦 Installing nnU-Net...
📦 Installing API server dependencies...
📦 Installing ML dependencies...

✅ Setup complete
   GPU available: True
   GPU: Tesla T4
   VRAM: 15.6 GB


CELL 2 — Load nnU-Net weights


In [2]:
import shutil, subprocess, glob, os

DRIVE_WEIGHTS = '/content/drive/MyDrive/mri/nnunet_weights/brats2020_weights.zip'
LOCAL_WEIGHTS = '/content/brats2020_weights.zip'

if os.path.exists(DRIVE_WEIGHTS):
    print("📦 Copying weights from Drive...")
    shutil.copy(DRIVE_WEIGHTS, LOCAL_WEIGHTS)
    print("✅ Weights copied from Drive")
else:
    print("📦 Downloading from Zenodo (~90 sec)...")
    ZIP_URL = "https://zenodo.org/records/4635763/files/Task082_nnUNetTrainerV2BraTSRegions_DA4_BN_BD__nnUNetPlansv2.1_bs5_5fold.zip"
    subprocess.run(['wget', '-q', '--show-progress', '-O', LOCAL_WEIGHTS, ZIP_URL], check=True)
    os.makedirs(os.path.dirname(DRIVE_WEIGHTS), exist_ok=True)
    shutil.copy(LOCAL_WEIGHTS, DRIVE_WEIGHTS)
    print("✅ Downloaded and saved to Drive for future sessions")

print("📦 Installing model into nnU-Net...")
subprocess.run(['nnUNet_install_pretrained_model_from_zip', LOCAL_WEIGHTS], check=True)

print("🔧 Patching nnU-Net for PyTorch 2.6...")
nnunet_file = '/usr/local/lib/python3.12/dist-packages/nnunet/training/model_restore.py'
if os.path.exists(nnunet_file):
    subprocess.run([
        'sed', '-i',
        "s/torch.load(i, map_location=torch.device('cpu'))/torch.load(i, map_location=torch.device('cpu'), weights_only=False)/g",
        nnunet_file
    ], check=False)

model_files = glob.glob('/content/nnunet_results/nnUNet/3d_fullres/Task082_BraTS2020/*/fold_*/model_final_checkpoint.model')
print(f"\n✅ nnU-Net ready — {len(model_files)} checkpoint(s) found")

📦 Copying weights from Drive...
✅ Weights copied from Drive
📦 Installing model into nnU-Net...
🔧 Patching nnU-Net for PyTorch 2.6...

✅ nnU-Net ready — 5 checkpoint(s) found


CELL 3 — Load MedGemma


In [3]:
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("✅ Logged into HuggingFace")

MODEL_ID = "google/medgemma-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("📦 Loading MedGemma processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, token=hf_token)

print("📦 Loading MedGemma model (4-bit, ~3 min first time)...")
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=hf_token,
)

print(f"\n✅ MedGemma loaded")
print(f"   VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

✅ Logged into HuggingFace
📦 Loading MedGemma processor...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]


✅ MedGemma loaded
   VRAM used: 3.4 GB


CELL 4 — Pipeline functions


In [4]:
import os, shutil, gzip, subprocess, uuid, base64
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from PIL import Image
from io import BytesIO
import torch


def _smart_load(path):
    with open(path, 'rb') as f:
        magic = f.read(2)
    if magic == b'\x1f\x8b':
        return nib.load(path)
    tmp = '/tmp/' + str(uuid.uuid4()) + '.nii'
    shutil.copy(path, tmp)
    img = nib.load(tmp)
    img.get_fdata()
    return img


def _copy_or_gzip(src, dst):
    with open(src, 'rb') as f:
        magic = f.read(2)
    if magic == b'\x1f\x8b':
        shutil.copy(src, dst)
    else:
        with open(src, 'rb') as fi, gzip.open(dst, 'wb', compresslevel=1) as fo:
            shutil.copyfileobj(fi, fo)


def segment_mri(flair_path, t1_path, t1ce_path, t2_path, output_name="patient"):
    session_id = uuid.uuid4().hex[:8]
    in_dir  = f'/content/api_uploads/seg_{session_id}'
    out_dir = f'/content/api_uploads/out_{session_id}'
    shutil.rmtree(in_dir,  ignore_errors=True)
    shutil.rmtree(out_dir, ignore_errors=True)
    os.makedirs(in_dir,  exist_ok=True)
    os.makedirs(out_dir, exist_ok=True)

    _copy_or_gzip(flair_path, os.path.join(in_dir, f'{output_name}_0000.nii.gz'))
    _copy_or_gzip(t1_path,    os.path.join(in_dir, f'{output_name}_0001.nii.gz'))
    _copy_or_gzip(t1ce_path,  os.path.join(in_dir, f'{output_name}_0002.nii.gz'))
    _copy_or_gzip(t2_path,    os.path.join(in_dir, f'{output_name}_0003.nii.gz'))

    print(f"  🧠 Running nnU-Net segmentation...")
    result = subprocess.run([
        'nnUNet_predict',
        '-i', in_dir, '-o', out_dir,
        '-t', 'Task082_BraTS2020',
        '-m', '3d_fullres',
        '-tr', 'nnUNetTrainerV2BraTSRegions_DA4_BN_BD',
        '-p', 'nnUNetPlansv2.1_bs5',
        '-f', '0',
    ], capture_output=True, text=True)

    if result.returncode != 0:
        print(f"  ❌ Error: {result.stderr[-500:]}")
        return {'success': False, 'error': result.stderr[-500:]}

    seg_path = os.path.join(out_dir, f'{output_name}.nii.gz')
    if not os.path.exists(seg_path):
        return {'success': False, 'error': 'No output file created'}

    seg_img          = _smart_load(seg_path)
    seg              = seg_img.get_fdata()
    n_tumor          = int((seg > 0).sum())
    voxel_vol_cm3    = float(np.prod(np.abs(np.diag(seg_img.affine)[:3]))) / 1000.0
    tumor_volume_cm3 = round(n_tumor * voxel_vol_cm3, 2)

    print(f"  ✅ Done — {n_tumor:,} tumor voxels ({tumor_volume_cm3} cm³)")

    return {
        'success':          n_tumor > 100,
        'seg_path':         seg_path,
        'in_dir':           in_dir,
        'tumor_voxels':     n_tumor,
        'tumor_volume_cm3': tumor_volume_cm3,
    }


def extract_tumor_facts(seg_path):
    LABEL_MAP = {1: 'enhancing', 2: 'necrotic', 3: 'edema'}

    img    = _smart_load(seg_path)
    seg    = img.get_fdata()
    affine = img.affine

    voxel_dims_mm    = np.abs(np.diag(affine)[:3])
    voxel_volume_cm3 = float(np.prod(voxel_dims_mm)) / 1000.0
    total_voxels     = int((seg > 0).sum())
    tumor_present    = total_voxels > 100

    facts = {"tumor_present": tumor_present}

    if not tumor_present:
        facts["summary"] = "No significant tumor detected."
        return facts

    subregions = {}
    for label_id, name in LABEL_MAP.items():
        n = int((seg == label_id).sum())
        subregions[name] = {"voxels": n, "volume_cm3": round(n * voxel_volume_cm3, 2)}
    facts["subregions"]       = subregions
    total_vol                 = round(sum(s["volume_cm3"] for s in subregions.values()), 2)
    facts["total_volume_cm3"] = total_vol

    if total_vol > 0:
        facts["composition_pct"] = {
            name: round(100 * subregions[name]["volume_cm3"] / total_vol, 1)
            for name in LABEL_MAP.values()
        }

    # Location
    tumor_coords   = np.argwhere(seg > 0)
    centroid_vox   = tumor_coords.mean(axis=0)
    centroid_world = nib.affines.apply_affine(affine, centroid_vox)
    orientation    = nib.aff2axcodes(affine)
    x_code         = orientation[0]
    midline_x      = seg.shape[0] / 2

    if x_code == 'L':
        hemisphere = "left" if centroid_vox[0] < midline_x else "right"
    elif x_code == 'R':
        hemisphere = "right" if centroid_vox[0] < midline_x else "left"
    else:
        hemisphere = "unknown"

    y_norm = centroid_vox[1] / seg.shape[1]
    y_code = orientation[1]
    if y_code == 'P':
        ap = "anterior" if y_norm < 0.33 else ("posterior" if y_norm > 0.66 else "central")
    elif y_code == 'A':
        ap = "anterior" if y_norm > 0.66 else ("posterior" if y_norm < 0.33 else "central")
    else:
        ap = "unknown"

    z_norm = centroid_vox[2] / seg.shape[2]
    si     = "inferior" if z_norm < 0.30 else ("superior" if z_norm > 0.70 else "mid-axial")

    loc_str    = f"{hemisphere} cerebral hemisphere"
    qualifiers = [q for q in [ap, si] if q not in ("central", "mid-axial", "unknown")]
    if qualifiers:
        loc_str += f" ({', '.join(qualifiers)} aspect)"

    facts["location"] = {"hemisphere": hemisphere, "description": loc_str}

    # Shape
    bbox_min    = tumor_coords.min(axis=0)
    bbox_max    = tumor_coords.max(axis=0)
    extent_mm   = (bbox_max - bbox_min + 1) * voxel_dims_mm
    approx_diam = round(2 * (3 * total_vol * 1000 / (4 * np.pi)) ** (1/3), 1)

    facts["shape"] = {
        "max_extent_mm":                 round(float(extent_mm.max()), 1),
        "approx_equivalent_diameter_mm": approx_diam,
    }

    # Flags
    flags = []
    if subregions['necrotic']['volume_cm3']  > 1.0: flags.append("central_necrosis_present")
    if subregions['enhancing']['volume_cm3'] > 1.0: flags.append("enhancing_component_present")
    if total_vol > 30:  flags.append("large_lesion")
    elif total_vol < 5: flags.append("small_lesion")
    if facts["shape"]["max_extent_mm"] > 60: flags.append("possible_mass_effect")
    if subregions['necrotic']['volume_cm3'] > 1.0 and subregions['enhancing']['volume_cm3'] > 1.0:
        flags.append("ring_enhancement_pattern")
    facts["clinical_flags"] = flags

    facts["summary"] = (
        f"A tumor of approximately {total_vol} cm³ in the {loc_str}. "
        f"Composition: {facts['composition_pct']['enhancing']}% enhancing, "
        f"{facts['composition_pct']['necrotic']}% necrotic, "
        f"{facts['composition_pct']['edema']}% edematous."
    )
    return facts


print("✅ All pipeline functions ready")

✅ All pipeline functions ready


In [5]:
import os, json
import numpy as np

# ── CONFIG ──────────────────────────────────────────────────
TEST_FOLDER_BASE = '/content/drive/MyDrive/mri/brats2020_mri/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
PATIENTS         = ['BraTS20_Training_001', 'BraTS20_Training_002', 'BraTS20_Training_003',
                    'BraTS20_Training_004', 'BraTS20_Training_005']

def find_modality(folder, keyword, exclude=None):
    for f in sorted(os.listdir(folder)):
        fl = f.lower()
        if keyword in fl and '_seg' not in fl:
            if exclude and exclude in fl: continue
            return os.path.join(folder, f)
    return None

def validate_report(report, facts):
    """14-point grounding validator"""
    score = 0
    checks = {}

    # 1. Total volume mentioned
    vol_str = str(facts['total_volume_cm3'])
    checks['total_volume']     = vol_str in report
    # 2. Location mentioned
    checks['location']         = facts['location']['hemisphere'] in report.lower()
    # 3. Enhancing volume
    en_vol = str(facts['subregions']['enhancing']['volume_cm3'])
    checks['enhancing_volume'] = en_vol in report
    # 4. Necrotic volume
    ne_vol = str(facts['subregions']['necrotic']['volume_cm3'])
    checks['necrotic_volume']  = ne_vol in report
    # 5. Edema volume
    ed_vol = str(facts['subregions']['edema']['volume_cm3'])
    checks['edema_volume']     = ed_vol in report
    # 6. Enhancing pct
    en_pct = str(facts['composition_pct']['enhancing'])
    checks['enhancing_pct']    = en_pct in report
    # 7. Necrotic pct
    ne_pct = str(facts['composition_pct']['necrotic'])
    checks['necrotic_pct']     = ne_pct in report
    # 8. Edema pct
    ed_pct = str(facts['composition_pct']['edema'])
    checks['edema_pct']        = ed_pct in report
    # 9. Max extent
    ext = str(facts['shape']['max_extent_mm'])
    checks['max_extent']       = ext in report
    # 10. TECHNIQUE section
    checks['technique_section']   = 'TECHNIQUE' in report.upper()
    # 11. FINDINGS section
    checks['findings_section']    = 'FINDINGS' in report.upper()
    # 12. IMPRESSION section
    checks['impression_section']  = 'IMPRESSION' in report.upper()
    # 13. RECOMMENDATIONS section
    checks['recommendations']     = 'RECOMMENDATION' in report.upper()
    # 14. Ring enhancement flag
    if 'ring_enhancement_pattern' in facts.get('clinical_flags', []):
        checks['ring_enhancement'] = any(k in report.lower() for k in ['ring', 'gbm', 'glioblastoma', 'high-grade'])
    else:
        checks['ring_enhancement'] = True  # not applicable, pass by default

    score = sum(checks.values())
    return score, checks

# ── RUN EVALUATION ──────────────────────────────────────────
print("="*60)
print("  RADIOS MRI EVALUATION — 5 PATIENTS")
print("="*60)

all_grounding_scores = []
all_results = []

for patient in PATIENTS:
    folder = os.path.join(TEST_FOLDER_BASE, patient)
    if not os.path.exists(folder):
        print(f"⚠️  {patient} not found, skipping")
        continue

    print(f"\n📋 Processing {patient}...")

    flair = find_modality(folder, 'flair')
    t1ce  = find_modality(folder, 't1ce')
    t1    = find_modality(folder, 't1', exclude='t1ce')
    t2    = find_modality(folder, 't2')

    # Segmentation
    seg_result = segment_mri(flair, t1, t1ce, t2, output_name=patient.lower())
    if not seg_result['success']:
        print(f"  ❌ Segmentation failed")
        continue

    # Facts
    facts = extract_tumor_facts(seg_result['seg_path'])

    # Build overlay
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
    from PIL import Image
    from io import BytesIO
    import base64

    seg_data   = _smart_load(seg_result['seg_path']).get_fdata()
    flair_data = _smart_load(flair).get_fdata()
    best_slice = (seg_data.shape[2] // 2 if (seg_data > 0).sum() == 0
                  else int(np.argmax((seg_data > 0).sum(axis=(0,1)))))

    seg_colors = np.zeros((5,4))
    seg_colors[1] = [1,1,0,0.7]
    seg_colors[2] = [1,0,0,0.6]
    seg_colors[3] = [0,1,0,0.5]
    cmap = ListedColormap(seg_colors)

    fig, ax = plt.subplots(figsize=(6,6))
    ax.imshow(flair_data[:,:,best_slice].T, cmap='gray', origin='lower')
    ax.imshow(seg_data[:,:,best_slice].T, cmap=cmap, vmin=0, vmax=4, origin='lower')
    ax.axis('off')
    buf = BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    overlay_pil = Image.open(buf).convert("RGB")

    # MedGemma report
    fact_summary = f"""DETERMINISTIC FINDINGS:
- Total volume: {facts['total_volume_cm3']} cm³
- Location: {facts['location']['description']}
- Enhancing: {facts['subregions']['enhancing']['volume_cm3']} cm³ ({facts['composition_pct']['enhancing']}%)
- Necrotic:  {facts['subregions']['necrotic']['volume_cm3']} cm³ ({facts['composition_pct']['necrotic']}%)
- Edema:     {facts['subregions']['edema']['volume_cm3']} cm³ ({facts['composition_pct']['edema']}%)
- Max extent: {facts['shape']['max_extent_mm']} mm
- Flags: {', '.join(facts['clinical_flags'])}"""

    messages = [
        {"role": "system", "content": [{"type": "text", "text":
            "You are an expert neuroradiologist. Base your report ONLY on the "
            "deterministic findings provided. Do not invent values."}]},
        {"role": "user", "content": [
            {"type": "image", "image": overlay_pil},
            {"type": "text", "text": f"""{fact_summary}
Write a structured report:
**TECHNIQUE** — brief description.
**FINDINGS** — use ALL exact numbers provided.
**IMPRESSION** — if ring-enhancement present, mention GBM/high-grade glioma.
**RECOMMENDATIONS** — next steps.
250-350 words."""},
        ]},
    ]

    inputs    = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=700,
                                do_sample=False, repetition_penalty=1.05)
    report = processor.decode(output[0][input_len:], skip_special_tokens=True)

    # Validate
    score, checks = validate_report(report, facts)
    all_grounding_scores.append(score)

    print(f"  ✅ Grounding score: {score}/14 ({round(score/14*100, 1)}%)")
    print(f"  📊 Checks passed: {[k for k,v in checks.items() if v]}")
    print(f"  ❌ Checks failed: {[k for k,v in checks.items() if not v]}")

    all_results.append({
        'patient': patient,
        'grounding_score': score,
        'grounding_pct': round(score/14*100, 1),
        'tumor_volume': facts['total_volume_cm3'],
        'checks': checks
    })

# ── SUMMARY ─────────────────────────────────────────────────
print("\n" + "="*60)
print("  EVALUATION SUMMARY")
print("="*60)
avg_grounding = np.mean(all_grounding_scores)
print(f"  Patients evaluated:       {len(all_grounding_scores)}")
print(f"  Avg grounding score:      {avg_grounding:.1f}/14")
print(f"  Avg grounding accuracy:   {avg_grounding/14*100:.1f}%")
print(f"  Min grounding score:      {min(all_grounding_scores)}/14")
print(f"  Max grounding score:      {max(all_grounding_scores)}/14")
print("="*60)

  RADIOS MRI EVALUATION — 5 PATIENTS

📋 Processing BraTS20_Training_001...
  🧠 Running nnU-Net segmentation...
  ✅ Done — 164,569 tumor voxels (164.57 cm³)
  ✅ Grounding score: 14/14 (100.0%)
  📊 Checks passed: ['total_volume', 'location', 'enhancing_volume', 'necrotic_volume', 'edema_volume', 'enhancing_pct', 'necrotic_pct', 'edema_pct', 'max_extent', 'technique_section', 'findings_section', 'impression_section', 'recommendations', 'ring_enhancement']
  ❌ Checks failed: []

📋 Processing BraTS20_Training_002...
  🧠 Running nnU-Net segmentation...
  ✅ Done — 22,638 tumor voxels (22.64 cm³)
  ✅ Grounding score: 14/14 (100.0%)
  📊 Checks passed: ['total_volume', 'location', 'enhancing_volume', 'necrotic_volume', 'edema_volume', 'enhancing_pct', 'necrotic_pct', 'edema_pct', 'max_extent', 'technique_section', 'findings_section', 'impression_section', 'recommendations', 'ring_enhancement']
  ❌ Checks failed: []

📋 Processing BraTS20_Training_003...
  🧠 Running nnU-Net segmentation...
  ✅ Don

CELL 5 — Quick test (optional but recommended)


In [6]:
# Tests the full pipeline on a BraTS patient from your Drive
# before starting the server. Run this to make sure everything works.

TEST_FOLDER = '/content/drive/MyDrive/mri/brats2020_mri/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_002'

def find_modality(folder, keyword, exclude=None):
    for f in sorted(os.listdir(folder)):
        fl = f.lower()
        if keyword in fl and '_seg' not in fl:
            if exclude and exclude in fl:
                continue
            return os.path.join(folder, f)
    return None

flair = find_modality(TEST_FOLDER, 'flair')
t1ce  = find_modality(TEST_FOLDER, 't1ce')
t1    = find_modality(TEST_FOLDER, 't1', exclude='t1ce')
t2    = find_modality(TEST_FOLDER, 't2')

print(f"FLAIR : {os.path.basename(flair)}")
print(f"T1    : {os.path.basename(t1)}")
print(f"T1ce  : {os.path.basename(t1ce)}")
print(f"T2    : {os.path.basename(t2)}\n")

result = segment_mri(flair, t1, t1ce, t2, output_name="test_patient")

if result['success']:
    facts = extract_tumor_facts(result['seg_path'])
    print(f"\n✅ TEST PASSED")
    print(f"   {facts['summary']}")
    print(f"   Flags: {facts['clinical_flags']}")
else:
    print(f"\n❌ TEST FAILED: {result.get('error', '')[:300]}")

FLAIR : BraTS20_Training_002_flair.nii
T1    : BraTS20_Training_002_t1.nii
T1ce  : BraTS20_Training_002_t1ce.nii
T2    : BraTS20_Training_002_t2.nii

  🧠 Running nnU-Net segmentation...
  ✅ Done — 22,640 tumor voxels (22.64 cm³)

✅ TEST PASSED
   A tumor of approximately 22.64 cm³ in the left cerebral hemisphere. Composition: 63.1% enhancing, 35.2% necrotic, 1.7% edematous.
   Flags: ['central_necrosis_present', 'enhancing_component_present', 'ring_enhancement_pattern']


CELL 6 — Start server (keep running)


In [7]:
import nest_asyncio, threading, time, uuid
import uvicorn
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pyngrok import ngrok, conf

nest_asyncio.apply()

# ── PASTE YOUR NGROK AUTHTOKEN BELOW ────────────────────────
NGROK_AUTHTOKEN = "3DSjZIHEHsnEGzteKbGD3ZzUWyC_6Qqw9z8f3AX4TYd9ZP6Qa"
NGROK_DOMAIN    = "headband-cheese-frosting.ngrok-free.dev"
# ────────────────────────────────────────────────────────────

conf.get_default().auth_token = NGROK_AUTHTOKEN

app = FastAPI(title="RADIOS MRI Backend")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/analyze-mri")
async def analyze_mri(
    flair: UploadFile = File(...),
    t1:    UploadFile = File(...),
    t1ce:  UploadFile = File(...),
    t2:    UploadFile = File(...),
):
    session_id = uuid.uuid4().hex[:8]
    upload_dir = f'/content/api_uploads/session_{session_id}'
    os.makedirs(upload_dir, exist_ok=True)

    try:
        # Save files
        saved = {}
        for mod_name, file_obj in [('flair',flair),('t1',t1),('t1ce',t1ce),('t2',t2)]:
            dest = os.path.join(upload_dir, file_obj.filename or f'{mod_name}.nii.gz')
            with open(dest, 'wb') as f:
                f.write(await file_obj.read())
            saved[mod_name] = dest
        print(f"\n[{session_id}] ✅ Files received")

        # Segmentation
        print(f"[{session_id}] 🧠 Running nnU-Net...")
        seg_result = segment_mri(saved['flair'], saved['t1'], saved['t1ce'], saved['t2'],
                                 output_name=f'api_{session_id}')
        if not seg_result.get('success'):
            raise HTTPException(status_code=500,
                detail=f"Segmentation failed: {seg_result.get('error','Unknown')}")

        # Facts
        facts = extract_tumor_facts(seg_result['seg_path'])
        print(f"[{session_id}] ✅ Facts extracted — {facts['total_volume_cm3']} cm³")

        # Overlay image
        seg_data   = _smart_load(seg_result['seg_path']).get_fdata()
        flair_data = _smart_load(saved['flair']).get_fdata()
        best_slice = (seg_data.shape[2] // 2 if (seg_data > 0).sum() == 0
                      else int(np.argmax((seg_data > 0).sum(axis=(0,1)))))

        seg_colors = np.zeros((5, 4))
        seg_colors[1] = [1.0, 1.0, 0.0, 0.7]
        seg_colors[2] = [1.0, 0.0, 0.0, 0.6]
        seg_colors[3] = [0.0, 1.0, 0.0, 0.5]
        cmap = ListedColormap(seg_colors)

        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        fig.patch.set_facecolor('#0d0d0d')
        for ax in axes: ax.axis('off')
        axes[0].imshow(flair_data[:,:,best_slice].T, cmap='gray', origin='lower')
        axes[0].set_title('Original FLAIR', color='white', fontsize=12)
        axes[1].imshow(flair_data[:,:,best_slice].T, cmap='gray', origin='lower')
        axes[1].imshow(seg_data[:,:,best_slice].T, cmap=cmap, vmin=0, vmax=4,
                       origin='lower', interpolation='nearest')
        axes[1].set_title('FLAIR + Tumor Segmentation', color='white', fontsize=12)
        plt.tight_layout()
        buf = BytesIO()
        plt.savefig(buf, format='png', dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
        plt.close(fig)
        buf.seek(0)
        overlay_b64 = base64.b64encode(buf.read()).decode('utf-8')
        overlay_pil = Image.open(BytesIO(base64.b64decode(overlay_b64))).convert("RGB")
        print(f"[{session_id}] ✅ Overlay built")

        # MedGemma report
        print(f"[{session_id}] 🤖 Running MedGemma...")
        fact_summary = f"""DETERMINISTIC FINDINGS:
- Total volume: {facts['total_volume_cm3']} cm³
- Location: {facts['location']['description']}
- Enhancing: {facts['subregions']['enhancing']['volume_cm3']} cm³ ({facts['composition_pct']['enhancing']}%)
- Necrotic:  {facts['subregions']['necrotic']['volume_cm3']} cm³ ({facts['composition_pct']['necrotic']}%)
- Edema:     {facts['subregions']['edema']['volume_cm3']} cm³ ({facts['composition_pct']['edema']}%)
- Max extent: {facts['shape']['max_extent_mm']} mm
- Flags: {', '.join(facts['clinical_flags'])}"""

        messages = [
            {"role": "system", "content": [{"type": "text", "text":
                "You are an expert neuroradiologist. Base your report ONLY on the "
                "deterministic findings provided. Do not invent values."}]},
            {"role": "user", "content": [
                {"type": "image", "image": overlay_pil},
                {"type": "text",  "text": f"""{fact_summary}

Image shows FLAIR with overlay (yellow=enhancing, red=necrotic, green=edema).

Write a structured report:
**TECHNIQUE** — brief description.
**FINDINGS** — use ALL exact numbers provided.
**IMPRESSION** — if ring-enhancement present, mention GBM/high-grade glioma.
**RECOMMENDATIONS** — next steps.
250-350 words."""},
            ]},
        ]

        inputs    = processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[-1]

        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=700,
                                    do_sample=False, repetition_penalty=1.05)

        report = processor.decode(output[0][input_len:], skip_special_tokens=True)
        print(f"[{session_id}] ✅ Report done")

        shutil.rmtree(upload_dir, ignore_errors=True)

        return JSONResponse({
            "success":          True,
            "overlay_image":    overlay_b64,
            "report":           report,
            "tumor_volume_cm3": facts['total_volume_cm3'],
            "location":         facts['location']['description'],
            "clinical_flags":   facts['clinical_flags'],
            "subregions":       facts['subregions'],
        })

    except HTTPException:
        raise
    except Exception as e:
        shutil.rmtree(upload_dir, ignore_errors=True)
        print(f"[{session_id}] ❌ Error: {e}")
        raise HTTPException(status_code=500, detail=str(e))


# Start server + tunnel
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

tunnel = ngrok.connect(addr=8000, domain=NGROK_DOMAIN, proto="http")

print("\n" + "="*55)
print("  ✅  RADIOS BACKEND IS LIVE")
print(f"  🌐  URL: https://{NGROK_DOMAIN}")
print(f"  🔬  POST https://{NGROK_DOMAIN}/analyze-mri")
print(f"  💚  GET  https://{NGROK_DOMAIN}/health")
print("="*55)
print("  ⚠️   Keep this cell running.")
print("="*55 + "\n")

threading.Thread(target=run_server, daemon=True).start()

while True:
    time.sleep(300)
    print("⏳ Backend still running...")


  ✅  RADIOS BACKEND IS LIVE
  🌐  URL: https://headband-cheese-frosting.ngrok-free.dev
  🔬  POST https://headband-cheese-frosting.ngrok-free.dev/analyze-mri
  💚  GET  https://headband-cheese-frosting.ngrok-free.dev/health
  ⚠️   Keep this cell running.

⏳ Backend still running...
⏳ Backend still running...
⏳ Backend still running...
⏳ Backend still running...

[da349a5e] ✅ Files received
[da349a5e] 🧠 Running nnU-Net...
  🧠 Running nnU-Net segmentation...
  ✅ Done — 263,202 tumor voxels (263.2 cm³)
[da349a5e] ✅ Facts extracted — 263.2 cm³
[da349a5e] ✅ Overlay built
[da349a5e] 🤖 Running MedGemma...
[da349a5e] ✅ Report done
⏳ Backend still running...
⏳ Backend still running...
⏳ Backend still running...


KeyboardInterrupt: 

In [2]:
!pip uninstall jax jaxlib -y -q
!pip install transformers==4.36.2 accelerate==0.27.0 -q
!pip install bitsandbytes>=0.45.0 -q --upgrade
print("✅ bitsandbytes upgraded")
print("✅ Done. Runtime → Restart Session, then run cells 2 onwards.")

✅ bitsandbytes upgraded
✅ Done. Runtime → Restart Session, then run cells 2 onwards.


In [1]:
from google.colab import files
from PIL import Image

uploaded = files.upload()
image_path = list(uploaded.keys())[0]
image = Image.open(image_path).convert("RGB")
image.thumbnail((512, 512))
image.save("/content/xray_input.png")  # save to disk for next step
print(f"✅ Image saved: {image.size}")

Saving CXR3_IM-1384-1001.png to CXR3_IM-1384-1001 (6).png
✅ Image saved: (512, 512)


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig
import json
from PIL import Image

CHEX_MODEL_ID = "StanfordAIMI/CheXagent-8b"

print("📦 Loading CheXagent in 8-bit...")
chex_processor = AutoProcessor.from_pretrained(CHEX_MODEL_ID, trust_remote_code=True)

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

chex_model = AutoModelForCausalLM.from_pretrained(
    CHEX_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
chex_model.eval()
print(f"✅ CheXagent loaded — VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

image = Image.open("/content/xray_input.png").convert("RGB")

def ask_chexagent(question, image):
    prompt = f" USER: <s>{question} ASSISTANT: <s>"
    inputs = chex_processor(images=image, text=prompt, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        output = chex_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            pad_token_id=chex_processor.tokenizer.eos_token_id,
        )
    text = chex_processor.decode(output[0], skip_special_tokens=True)
    if "ASSISTANT:" in text:
        text = text.split("ASSISTANT:")[-1].strip()
    return text.replace("<s>","").replace("</s>","").strip()

print("\n🧠 Extracting findings with CheXagent...")
findings    = ask_chexagent("Describe all abnormalities and findings visible in this chest X-ray including lung fields, heart size, mediastinum, and pleural spaces.", image)
pathologies = ask_chexagent("List all pathologies and diseases you can identify in this chest X-ray.", image)
impression  = ask_chexagent("What is the radiological impression and most likely diagnosis for this chest X-ray?", image)

print(f"\n📋 CheXagent findings:")
print(f"   Findings:    {findings}")
print(f"   Pathologies: {pathologies}")
print(f"   Impression:  {impression}")

# Save to disk
with open("/content/chex_findings.json", "w") as f:
    json.dump({"findings": findings, "pathologies": pathologies, "impression": impression}, f)

# Unload
print("\n🗑️ Unloading CheXagent...")
del chex_model, chex_processor
torch.cuda.empty_cache()
import gc; gc.collect()
print(f"✅ VRAM freed — now: {torch.cuda.memory_allocated()/1e9:.1f} GB")

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


📦 Loading CheXagent in 8-bit...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ CheXagent loaded — VRAM: 9.2 GB

🧠 Extracting findings with CheXagent...


/root/.cache/huggingface/modules/transformers_modules/StanfordAIMI/CheXagent-8b/4934e91451945c8218c267aae9c34929a7677829/processing_chexagent.py:86: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  [torch.tensor(pixel_values) for pixel_values in encoding_image_processor["pixel_values"]]



📋 CheXagent findings:
   Findings:    Pleural effusion
   Pathologies: Lung cancer
   Impression:  Lung Opacity

🗑️ Unloading CheXagent...
✅ VRAM freed — now: 0.0 GB


In [2]:
!pip install transformers -q --upgrade
!pip install accelerate -q
print("✅ Done, now run Cell 5 again")
print("✅ Now go to Runtime → Restart Session")

✅ Done, now run Cell 5 again
✅ Now go to Runtime → Restart Session


In [1]:
import subprocess
subprocess.run(["pip", "install", "accelerate", "bitsandbytes", "-q"], check=True)

import torch, json
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# Load findings from CheXagent
with open("/content/chex_findings.json") as f:
    chex = json.load(f)
findings    = chex["findings"]
pathologies = chex["pathologies"]
impression  = chex["impression"]
image = Image.open("/content/xray_input.png").convert("RGB")

print(f"📋 Loaded findings:")
print(f"   Findings:    {findings}")
print(f"   Pathologies: {pathologies}")
print(f"   Impression:  {impression}\n")

MEDGEMMA_ID = "google/medgemma-4b-it"
print("📦 Loading MedGemma 4B...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
mg_processor = AutoProcessor.from_pretrained(MEDGEMMA_ID, token=hf_token)
mg_model = AutoModelForImageTextToText.from_pretrained(
    MEDGEMMA_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    token=hf_token,
)
mg_model.eval()
print(f"✅ MedGemma loaded — VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

fact_summary = f"""CHEXAGENT DETECTED FINDINGS:
- Findings:    {findings}
- Pathologies: {pathologies}
- Impression:  {impression}
"""

messages = [
    {"role": "system", "content": [{"type": "text", "text": (
        "You are an expert radiologist writing a formal chest X-ray report. "
        "Base your report on the detected findings provided and the X-ray image. "
        "Do not invent findings not supported by the data."
    )}]},
    {"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": f"""{fact_summary}

Write a detailed structured radiology report with these exact sections:

**TECHNIQUE**
Describe the imaging technique and patient positioning.

**FINDINGS**
Describe in detail: lung fields, heart size and contour, mediastinum,
pleural spaces, diaphragm, bones, and soft tissues. Reference the
detected pathologies above with clinical detail.

**IMPRESSION**
Summarize the key findings and their clinical significance clearly.

**RECOMMENDATIONS**
Suggest appropriate clinical next steps, follow-up imaging, or specialist referral.

Write 300-400 words. Use formal radiology language."""},
    ]},
]

print("\n🧠 MedGemma generating full report...")
inputs = mg_processor.apply_chat_template(
    messages, add_generation_prompt=True,
    tokenize=True, return_dict=True, return_tensors="pt",
).to(mg_model.device)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    output = mg_model.generate(
        **inputs,
        max_new_tokens=700,
        do_sample=False,
        repetition_penalty=1.05,
    )

report = mg_processor.decode(output[0][input_len:], skip_special_tokens=True)

print("\n" + "="*60)
print("       CHEST X-RAY REPORT")
print("="*60)
print(report)
print("="*60)

📋 Loaded findings:
   Findings:    Pleural effusion
   Pathologies: Lung cancer
   Impression:  Lung Opacity

📦 Loading MedGemma 4B...


ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`

In [4]:
!pip install accelerate==1.6.0 --force-reinstall -q
import importlib
import accelerate
importlib.reload(accelerate)
print(f"✅ accelerate version: {accelerate.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ImportError: cannot import name 'DataLoaderConfiguration' from 'accelerate.utils' (/usr/local/lib/python3.12/dist-packages/accelerate/utils/__init__.py)